In [1]:
import pandas as pd
import numpy as np
import joblib
import time
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, GridSearchCV, RandomizedSearchCV
from xgboost import XGBRegressor

X_train = pd.read_csv("../data/processed/X_train.csv")
y_train = pd.read_csv("../data/processed/y_train.csv").squeeze()

preprocessor = joblib.load("../models/preprocessor.pkl")
X_train_processed = preprocessor.transform(X_train)

cv = KFold(n_splits=5, shuffle=True, random_state=42)

In [2]:
from sklearn.model_selection import cross_val_score

ridge_before = Ridge(alpha=1.0, random_state=42)
ridge_before_score = -cross_val_score(ridge_before, X_train_processed, y_train,
                                        cv=cv, scoring='neg_root_mean_squared_error').mean()

xgb_before = XGBRegressor(n_estimators=200, max_depth=4, learning_rate=0.05,
                            n_jobs=-1, random_state=42, verbosity=0)
xgb_before_score = -cross_val_score(xgb_before, X_train_processed, y_train,
                                      cv=cv, scoring='neg_root_mean_squared_error').mean()

print(f"Ridge before tuning: {ridge_before_score:.4f}")
print(f"XGBoost before tuning: {xgb_before_score:.4f}")

Ridge before tuning: 0.1195
XGBoost before tuning: 0.1201


In [3]:
ridge_param_grid = {
    'alpha': [0.01, 0.1, 0.5, 1.0, 5.0, 10.0, 20.0, 50.0, 100.0]
}

ridge_grid = GridSearchCV(
    Ridge(random_state=42),
    param_grid=ridge_param_grid,
    cv=cv,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

start = time.time()
ridge_grid.fit(X_train_processed, y_train)
ridge_tune_time = time.time() - start

print("Best alpha:", ridge_grid.best_params_)
print("Best CV RMSE:", -ridge_grid.best_score_)
print(f"Tuning time: {ridge_tune_time:.1f}s")

Best alpha: {'alpha': 20.0}
Best CV RMSE: 0.11404451046914779
Tuning time: 2.0s


In [4]:
xgb_param_dist = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [2, 3, 4, 5, 6],
    'learning_rate': [0.01, 0.03, 0.05, 0.1],
    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    'reg_lambda': [0.1, 1.0, 5.0, 10.0]
}

xgb_random = RandomizedSearchCV(
    XGBRegressor(random_state=42, n_jobs=-1, verbosity=0),
    param_distributions=xgb_param_dist,
    n_iter=40,
    cv=cv,
    scoring='neg_root_mean_squared_error',
    random_state=42,
    n_jobs=1  # outer loop stays 1 since XGBoost itself uses n_jobs=-1 internally
)

start = time.time()
xgb_random.fit(X_train_processed, y_train)
xgb_tune_time = time.time() - start

print("Best params:", xgb_random.best_params_)
print("Best CV RMSE:", -xgb_random.best_score_)
print(f"Tuning time: {xgb_tune_time:.1f}s")

Best params: {'subsample': 0.6, 'reg_lambda': 5.0, 'n_estimators': 500, 'max_depth': 2, 'learning_rate': 0.05, 'colsample_bytree': 1.0}
Best CV RMSE: 0.11696097212376563
Tuning time: 69.2s


In [5]:
comparison = pd.DataFrame({
    'Model': ['Ridge (before)', 'Ridge (tuned)', 'XGBoost (before)', 'XGBoost (tuned)'],
    'CV_RMSE': [ridge_before_score, -ridge_grid.best_score_,
                xgb_before_score, -xgb_random.best_score_]
})
print(comparison)

              Model   CV_RMSE
0    Ridge (before)  0.119533
1     Ridge (tuned)  0.114045
2  XGBoost (before)  0.120121
3   XGBoost (tuned)  0.116961


In [6]:
best_ridge = ridge_grid.best_estimator_
best_xgb = xgb_random.best_estimator_

joblib.dump(best_ridge, "../models/ridge_tuned.pkl")
joblib.dump(best_xgb, "../models/xgb_tuned.pkl")

['../models/xgb_tuned.pkl']

In [7]:
print(ridge_grid.best_params_)

{'alpha': 20.0}
